# Banana Harness in Colab, Molab, or an existing container

This notebook is a template. Only synthetic offline tasks run by default. Real benchmark runs require you to turn on the switches below. The package has not been published yet; install your locally built wheel or upload this repository first. Python 3.10+ and Linux are required.


In [ ]:
# Before publication, upload the repository or built wheel and use its path.
# %pip install /path/to/banana_harness-0.1.0-py3-none-any.whl
# After publication:
# %pip install "banana-harness[all]"


In [ ]:
from pathlib import Path
from banana_harness.cli import diagnostics, smoke

diagnostics()


In [ ]:
summary = smoke(Path("jobs/notebook-smoke"))
print(f"{summary['passed']}/{summary['tasks']} synthetic tasks passed")


## Runtime setup

The native path runs commands in the notebook host. The rootfs path uses OCI image files and udocker PRoot; it requires usable `ptrace` but no Docker daemon or root. Store the cache and jobs on persistent disk if you want to resume after runtime restarts. This does not isolate untrusted commands from your host.


In [ ]:
RUN_REAL_BENCHMARK = False
ALLOW_BENCHMARK_DOWNLOADS = False
INSTALL_ROOTFS_TOOLS = False
CACHE = Path(".banana/udocker").resolve()
OUTPUT = Path("jobs/notebook-benchmark")


In [ ]:
# An explicit install downloads udocker execution binaries, not benchmark tasks.
# Turn INSTALL_ROOTFS_TOOLS on only when you want that operation.
import os
import shutil
import subprocess

if INSTALL_ROOTFS_TOOLS:
    executable = shutil.which("udocker")
    if not executable:
        raise RuntimeError("Install banana-harness[rootfs] first")
    command = [executable] + (["--allow-root"] if os.geteuid() == 0 else []) + ["install"]
    subprocess.run(command, env={**os.environ, "UDOCKER_DIR": str(CACHE)}, check=True)
else:
    print("Rootfs tool installation skipped")


## Choose an existing task source

Use `swebench-verified` with a local official JSONL, `livecodebench-v6` with a local v6 JSONL, or a Harbor task directory for `swebench-pro`, `deepswe`, or `terminal-bench-2.1`. Pro uses `v2/tasks`; DeepSWE uses `tasks`; Terminal-Bench uses its dedicated 2.1 repository root. Model keys stay in the notebook process.


In [ ]:
BENCHMARK = "deepswe"
SOURCE = Path("/path/to/deep-swe/tasks")
MODEL = "provider/model"
# Set your provider credentials through notebook secrets or os.environ.
# Do not save credentials in this notebook.


In [ ]:
# Optional explicit fetch, disabled by default. This downloads task definitions.
from banana_harness.download import fetch

if RUN_REAL_BENCHMARK and ALLOW_BENCHMARK_DOWNLOADS and not SOURCE.exists():
    SOURCE = fetch(BENCHMARK, Path("datasets") / BENCHMARK, allow_downloads=True)
else:
    print("Task-definition download skipped")


In [ ]:
from banana_harness import RunConfig, Runner
from banana_harness.adapters import load_tasks, select_tasks
from banana_harness.agent import LiteLLMModel

if RUN_REAL_BENCHMARK:
    tasks = select_tasks(load_tasks(BENCHMARK, SOURCE), limit=1)
    runner = Runner(
        lambda: LiteLLMModel(MODEL),
        RunConfig(output=OUTPUT, cache=CACHE, allow_downloads=ALLOW_BENCHMARK_DOWNLOADS),
    )
    benchmark_summary = runner.run(tasks)
    print(benchmark_summary["passed"], "/", benchmark_summary["tasks"], "passed")
else:
    print("Real benchmark execution skipped")


## Results

Inspect `summary.json`, task `result.json`, and each attempt's `trajectory.json`, `model.patch`, and verifier artifacts. Repeating the same configuration resumes completed passes/failures. Host results record runtime differences and are not official leaderboard scores. The built-in LiveCodeBench grader is approximate; set `RunConfig(livecode_evaluator=Path("/path/to/LiveCodeBench"))` to use an existing upstream grader checkout.
